# 02 — Feature Engineering

**Objectif** : Préparer les données pour la modélisation. Enrichir le dataset 
avec des facteurs CO2 variables depuis la Base Carbone ADEME pour remplacer 
les constantes de Back-on-Track 2022.

**Inputs** : Base PostgreSQL `obrail_db`, `data/raw/Base_Carbone_V23.9.csv`  
**Outputs** : Dataset enrichi avec facteurs CO2 variables  
**Date** : Avril 2026

## Imports

In [6]:
import pandas as pd
from sqlalchemy import create_engine

## Chargement des données

In [7]:
df = pd.read_csv(
    '/Users/louisgardet/dev/python/OBRail-MSPR2/data/raw/environmental_impact.csv'
)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (3678, 18)


,route_name,origin,destination,service_type,route_name_simple,origin_country,destination_country,distance_km,operator,type,train_gco2_pkm,plane_gco2_pkm,train_co2_kg,plane_co2_kg,co2_savings_kg,savings_percent,emission_source,calculation_date
0,ICE 18,Hauptbahnhof,Berlin Gesundbrunnen,day,→ Berlin Gesundbrunnen,DE,DE,238.64,DB,day,14,144,3.34,34.36,31.02,90.3,Back-on-Track 2022,2026-03-22
1,IC 51,Hauptbahnhof,Dortmund Hbf,day,→ Dortmund,DE,DE,256.33,DB,day,14,144,3.59,36.91,33.32,90.3,Back-on-Track 2022,2026-03-22
2,ICE 28,Hamburg Altona S,Hauptbahnhof,day,Hamburg Altona S →,DE,DE,296.60,DB,day,14,144,4.15,42.71,38.56,90.3,Back-on-Track 2022,2026-03-22


## Extraction des facteurs CO2 ferroviaires (ADEME Base Carbone V23.9)

La Base Carbone contient 23 000+ entrées. On filtre uniquement les lignes 
transport ferroviaire avec une unité par passager-km.

In [8]:
df_ademe = pd.read_csv(
    '/Users/louisgardet/dev/python/OBRail-MSPR2/data/raw/Base_Carbone_V23.9.csv',
    sep=';', encoding='latin-1', low_memory=False
)

mask = (
    df_ademe['Tags français'].str.contains(
        'ferroviaire|TGV|TER|Intercit', case=False, na=False
    ) &
    df_ademe['Unité français'].str.contains(
        'passager|pass|pkm|voy', case=False, na=False
    )
)

df_rail = df_ademe[mask][[
    'Nom base français', 'Unité français',
    'Total poste non décomposé', 'Tags français'
]].copy()

df_rail['co2_pkm'] = df_rail['Total poste non décomposé'].str.replace(',', '.').astype(float)

summary = df_rail.groupby(
    df_rail['Nom base français'].str.extract(r'(TGV|TER|Intercités)')[0]
)['co2_pkm'].mean().reset_index()
summary.columns = ['train_type', 'co2_pkm_mean']
print(summary)

   train_type  co2_pkm_mean
0  Intercités      0.008797
1         TER      0.034818
2         TGV      0.002970


## Enrichissement du dataset

On applique les facteurs ADEME aux routes SNCF selon le type de service.
Les autres opérateurs (DB, ÖBB, SBB...) conservent le facteur Back-on-Track 
(0.014) faute de données équivalentes pour leurs réseaux nationaux.

| Type       | Facteur (kgCO2/pkm) | Mapping         |
|------------|---------------------|-----------------|
| TGV        | 0.00297             | SNCF + jour     |
| Intercités | 0.008797            | SNCF + nuit     |
| Autres     | 0.014               | Fallback        |

In [9]:
co2_map = {
    ('SNCF', 'day'):   0.00297,    # TGV
    ('SNCF', 'night'): 0.008797,   # Intercités
}
co2_default = 0.014  # Back-on-Track fallback

df['train_gco2_pkm_ademe'] = df.apply(
    lambda row: co2_map.get((row['operator'], row['service_type']), co2_default),
    axis=1
)

# Validation
print(df[df['operator'] == 'SNCF'].groupby('service_type')['train_gco2_pkm_ademe'].mean())
print(f"\nValeurs uniques : {sorted(df['train_gco2_pkm_ademe'].unique())}")

service_type
day      0.002970
night    0.008797
Name: train_gco2_pkm_ademe, dtype: float64

Valeurs uniques : [np.float64(0.00297), np.float64(0.008797), np.float64(0.014)]


## Conclusion

Le dataset enrichi contient désormais des facteurs CO2 variables pour les 
trains français (SNCF), remplaçant la constante uniforme de Back-on-Track 2022.

**Prochaine étape** : Utiliser `train_gco2_pkm_ademe` pour recalculer 
`co2_savings_kg` et en faire une vraie cible ML non triviale.